# Debugging a knot vector, live

`make_smoothing_spline` on the scipy `userknots` branch accepts user knots `t`.
The knots must be clamped: the first four and last four knots each equal.
This notebook traces the validation the way a user would debug it: start with
a good knot vector, read what the code assumed, then break the vector and
watch where it fails.

In [1]:
import numpy as np
import sympy
from skverify import to_sympy

# the branch's knot gate, verbatim semantics
# (scipy/interpolate/_bsplines.py, _make_smoothing_spline_user_knots)
def knot_gate(t):
    if t.ndim != 1 or np.any(t[1:] - t[:-1] < 0):
        raise ValueError("`t` must be a 1-D non-decreasing array")
    if not (t[0] == t[3] and t[-4] == t[-1]):
        raise ValueError("`t` must be clamped: the first 4 and last 4 "
                         "knots must each be equal")
    return np.sum(t[4:-4])   # past the gate: use the interior knots

k = 3
interior = np.array([1.3, 2.1, 2.9])
t_good = np.r_[[0.0] * (k + 1), interior, [4.0] * (k + 1)]
t_good

array([0. , 0. , 0. , 0. , 1.3, 2.1, 2.9, 4. , 4. , 4. , 4. ])

A valid clamped vector. Trace it. The certificate shows the formula
computed past the gate, and every branch the gate took becomes a stated
hypothesis:

In [2]:
out = to_sympy(knot_gate, t_good)
out.pretty()

'\nformula    = Sum(t[j + 4], (j, 0, 2))\nassumes[0] = Eq(t[0], t[3])\nassumes[1] = Eq(t[7], t[10])\nassumes[2] = Sum(Piecewise((1, t[j + 1] - t[j] < 0), (0, True)), (j, 0, 9)) <= 0'

Read the assumptions. The code checked three things and the certificate
wrote them down as mathematics:

In [3]:
from IPython.display import Math, display
for g in out.preconditions.args:
    display(Math(sympy.latex(g)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Two boundary equalities and one monotonicity condition (the count of
decreasing adjacent pairs is zero). Note what the code actually demands:
only $t_0 = t_3$, not all four equalities.

Now break the vector the way a real bug would: keep it sorted, lose the
boundary multiplicity.

In [4]:
t_bad = t_good.copy()
t_bad[0] -= 0.4     # still non-decreasing, no longer clamped
try:
    to_sympy(knot_gate, t_bad)
except ValueError as e:
    print(type(e).__name__ + ":", e)

ValueError: `t` must be clamped: the first 4 and last 4 knots must each be equal


The trace runs your real code, so your own error arrives untouched.
The certificate from the good run tells you exactly which hypothesis the bad
vector violates: $t_0 = t_3$ fails, since $t_0$ moved.

One question is left. The gate checks $t_0 = t_3$ only. Clamped means all
four equal. Is the check enough? Prove it: with the knots sorted, no vector
can pass the gate and stay unclamped.

In [5]:
from sympy.logic.inference import satisfiable

t0, t1, t2, t3 = sympy.symbols("t0 t1 t2 t3", real=True)

def eq(a, b):          # equality as two inequalities: pure linear arithmetic
    return sympy.And(a <= b, b <= a)

gate    = sympy.And(t0 <= t1, t1 <= t2, t2 <= t3, eq(t0, t3))
clamped = sympy.And(eq(t0, t1), eq(t1, t2), eq(t2, t3))

satisfiable(sympy.And(gate, sympy.Not(clamped)), use_lra_theory=True)

False

No model exists: passing the gate forces the clamped vector. The check
in the code is exactly sufficient.

**Takeaway.** The trace turned a code path into three stated hypotheses. The
failing run pointed at the one that broke. The solver proved the gate equals
the mathematical definition. None of this needed print statements.